# 05 · Analysis & Figures

Generates all paper figures and tables from the JSON results files.
Can be run locally (no GPU needed).

**Requires:** `results/main_comparison.json`, `results/generalization.json`,
`results/learning_curve.json`.

**Writes:** figures to `figures/`.

In [ ]:
!pip install -q matplotlib seaborn pandas numpy scikit-learn

## 1 · Configuration

In [ ]:
# ── Toggle this when running on Colab ──────────────────────────
USE_DRIVE  = False
DRIVE_PATH = '/content/drive/MyDrive/cs443-final-project'
# ───────────────────────────────────────────────────────────────

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path(DRIVE_PATH)
else:
    BASE = Path('.')

RESULTS_DIR = BASE / 'results'
FIGURES_DIR = BASE / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['figure.dpi'] = 150

# Consistent colours and display names
MODEL_DISPLAY = {
    'tfidf_logreg': 'TF-IDF + LogReg',
    'roberta-base':  'RoBERTa-base',
    'finbert':       'FinBERT',
    'haiku':         'Claude Haiku 4.5',
    'sonnet':        'Claude Sonnet 4.6',
}
MODEL_COLORS = {
    'tfidf_logreg': '#7f8c8d',
    'roberta-base':  '#3498db',
    'finbert':       '#2ecc71',
    'haiku':         '#e67e22',
    'sonnet':        '#9b59b6',
}
DISPLAY_ORDER = ['tfidf_logreg', 'roberta-base', 'finbert', 'haiku', 'sonnet']

ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

print('Config ready. Figures will be saved to:', FIGURES_DIR.resolve())

## 2 · Load Results

In [ ]:
def load_json(path):
    p = Path(path)
    if not p.exists():
        print(f'  WARNING: {path} not found — skipping.')
        return {}
    with open(p) as f:
        return json.load(f)

main_results = load_json(RESULTS_DIR / 'main_comparison.json')
gen_results  = load_json(RESULTS_DIR / 'generalization.json')
lc_results   = load_json(RESULTS_DIR / 'learning_curve.json')

print('main_comparison models :', list(main_results.keys()))
print('generalization models  :', list(gen_results.keys()))
print('learning_curve models  :', list(lc_results.keys()))

## 3 · Model Comparison Table

Accuracy and macro F1 on PhraseBank test and Twitter OOD for all five models.

In [ ]:
rows = []
for key in DISPLAY_ORDER:
    if key not in main_results:
        continue
    pb = main_results[key].get('phrasebank_test', {})
    tw = gen_results.get(key, {})
    row = {
        'Model':           MODEL_DISPLAY.get(key, key),
        'PB Acc':          round(pb.get('accuracy', float('nan')), 4),
        'PB Macro F1':     round(pb.get('macro_f1', float('nan')), 4),
        'TW Acc':          round(tw.get('accuracy', float('nan')), 4),
        'TW Macro F1':     round(tw.get('macro_f1', float('nan')), 4),
        'Cost / 1K (USD)': round(pb.get('cost_per_1k', 0.0), 4),
    }
    rows.append(row)

df = pd.DataFrame(rows).set_index('Model')
print('\nMain results table:')
print(df.to_string())

## 4 · Bar Chart: Model Comparison

In [ ]:
def bar_chart(results, metric_key, dataset_label, filename):
    keys    = [k for k in DISPLAY_ORDER if k in results]
    names   = [MODEL_DISPLAY.get(k, k) for k in keys]
    accs    = [results[k].get('phrasebank_test' if 'phrasebank_test' in results[k] else '', {}).get('accuracy', 0)
               for k in keys]
    f1s     = [results[k].get('phrasebank_test' if 'phrasebank_test' in results[k] else '', {}).get('macro_f1', 0)
               for k in keys]
    colors  = [MODEL_COLORS.get(k, '#95a5a6') for k in keys]

    x     = np.arange(len(names))
    width = 0.35
    fig, ax = plt.subplots(figsize=(9, 4.5))

    b1 = ax.bar(x - width / 2, accs, width, label='Accuracy', color=colors, alpha=0.85)
    b2 = ax.bar(x + width / 2, f1s,  width, label='Macro F1',
                color=colors, alpha=0.50, edgecolor=colors, linewidth=1.5)

    for bar in list(b1) + list(b2):
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.008,
                f'{h:.2f}', ha='center', va='bottom', fontsize=8)

    ax.set_ylabel('Score')
    ax.set_title(f'Model Comparison — {dataset_label}')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=15, ha='right')
    ax.set_ylim(0, 1.12)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', path)


# Reshape for the generic bar_chart helper
pb_flat  = {k: {'phrasebank_test': v['phrasebank_test']}
            for k, v in main_results.items() if 'phrasebank_test' in v}
gen_flat = {k: {'phrasebank_test': v}            # reuse key name for helper
            for k, v in gen_results.items()}

if pb_flat:
    bar_chart(pb_flat, 'macro_f1', 'PhraseBank Test', 'bar_phrasebank.png')
if gen_flat:
    bar_chart(gen_flat, 'macro_f1', 'Twitter OOD',    'bar_twitter.png')

## 5 · Confusion Matrix for Best Fine-tuned Model

Shows which classes are confused most often.

In [ ]:
LABEL_NAMES = ['negative', 'neutral', 'positive']


def plot_cm(cm_data, title, filename):
    arr = np.array(cm_data)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(arr, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    fig.tight_layout()
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', path)


# Pick best fine-tuned model by macro F1
ft_models = ['roberta-base', 'finbert']
best_ft = max(
    (k for k in ft_models if k in main_results),
    key=lambda k: main_results[k]['phrasebank_test'].get('macro_f1', 0),
    default=None,
)

if best_ft:
    cm = main_results[best_ft]['phrasebank_test']['confusion_matrix']
    display = MODEL_DISPLAY.get(best_ft, best_ft)
    plot_cm(cm, f'{display} — PhraseBank Test', f'cm_{best_ft}_phrasebank.png')

    if best_ft in gen_results and 'confusion_matrix' in gen_results[best_ft]:
        cm_ood = gen_results[best_ft]['confusion_matrix']
        plot_cm(cm_ood, f'{display} — Twitter OOD', f'cm_{best_ft}_twitter.png')
else:
    print('No fine-tuned model results found. Run notebook 02 first.')

## 6 · Learning Curve

Macro F1 vs training size for fine-tuned models. Horizontal dashed lines show
LLM zero-shot scores as reference.

In [ ]:
if not lc_results:
    print('No learning curve data. Run notebook 02 first.')
else:
    fig, ax = plt.subplots(figsize=(8, 5))

    for model_key, points in lc_results.items():
        sizes = [p['train_size'] for p in points]
        f1s   = [p['macro_f1']   for p in points]
        color   = MODEL_COLORS.get(model_key, '#333')
        display = MODEL_DISPLAY.get(model_key, model_key)
        ax.plot(sizes, f1s, 'o-', color=color, label=display, linewidth=2, markersize=6)

    # LLM reference lines
    for llm_key in ['haiku', 'sonnet']:
        if llm_key in main_results and 'phrasebank_test' in main_results[llm_key]:
            f1      = main_results[llm_key]['phrasebank_test']['macro_f1']
            color   = MODEL_COLORS.get(llm_key, '#333')
            display = MODEL_DISPLAY.get(llm_key, llm_key)
            ax.axhline(y=f1, color=color, linestyle='--', linewidth=1.5,
                       label=f'{display} (zero-shot)', alpha=0.8)

    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('Macro F1')
    ax.set_title('Learning Curve: Fine-tuned Models vs Zero-shot LLMs')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
    ax.set_ylim(0, 1.05)
    fig.tight_layout()

    path = FIGURES_DIR / 'learning_curve.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', path)

## 7 · Error Analysis

Find examples where fine-tuned models and LLMs disagree — these are the most
informative cases for qualitative discussion in the paper.

In [ ]:
# Reload the PhraseBank test split (same seed → same examples as all notebooks)
from datasets import load_dataset
from sklearn.model_selection import train_test_split

raw = load_dataset('gtfintechlab/financial_phrasebank_sentences_allagree')
ds  = raw['train']
if 'sentence' in ds.column_names:
    ds = ds.rename_column('sentence', 'text')

texts  = list(ds['text'])
labels = list(ds['label'])
_, tmp_texts, _, tmp_labels = train_test_split(
    texts, labels, test_size=0.30, random_state=42, stratify=labels
)
_, test_texts, _, test_labels = train_test_split(
    tmp_texts, tmp_labels, test_size=0.50, random_state=42, stratify=tmp_labels
)
print(f'Test set: {len(test_texts)} examples loaded.')

In [ ]:
import hashlib, re

HAIKU_ID  = 'claude-haiku-4-5-20251001'
SONNET_ID = 'claude-sonnet-4-6'


def load_llm_preds(model_id, dataset_name, texts):
    """Re-read cached LLM predictions for the test set."""
    cache_file = BASE / 'cache' / f'llm_{model_id}_{dataset_name}.jsonl'
    if not cache_file.exists():
        return None
    cache = {}
    with open(cache_file) as f:
        for line in f:
            e = json.loads(line)
            cache[e['key']] = e.get('parsed_label')
    preds = []
    for text in texts:
        key    = hashlib.sha256(text.encode()).hexdigest()
        parsed = cache.get(key)
        label  = {'negative': 0, 'neutral': 1, 'positive': 2}.get(parsed, 1)
        preds.append(label)
    return preds


haiku_preds  = load_llm_preds(HAIKU_ID,  'phrasebank_test', test_texts)
sonnet_preds = load_llm_preds(SONNET_ID, 'phrasebank_test', test_texts)

if haiku_preds:
    print('Haiku preds loaded:  {} examples'.format(len(haiku_preds)))
else:
    print('No Haiku cache found — run notebook 03 first.')
if sonnet_preds:
    print('Sonnet preds loaded: {} examples'.format(len(sonnet_preds)))
else:
    print('No Sonnet cache found — run notebook 03 first.')

In [ ]:
ID2LABEL = {0: 'negative', 1: 'neutral', 2: 'positive'}

if haiku_preds and sonnet_preds:
    # Find examples where Haiku and Sonnet disagree
    disagreements = [
        i for i in range(len(test_texts))
        if haiku_preds[i] != sonnet_preds[i]
    ]
    print(f'Haiku vs Sonnet disagreements: {len(disagreements)} / {len(test_texts)}')

    # Also find cases where both LLMs are wrong
    both_wrong = [
        i for i in range(len(test_texts))
        if haiku_preds[i] != test_labels[i] and sonnet_preds[i] != test_labels[i]
    ]
    print(f'Both LLMs wrong       : {len(both_wrong)} / {len(test_texts)}')

    print('\n' + '─' * 70)
    print('Sample disagreements (Haiku vs Sonnet):')
    print('─' * 70)
    shown = 0
    for i in disagreements[:20]:          # scan first 20 to find interesting ones
        true = test_labels[i]
        h    = haiku_preds[i]
        s    = sonnet_preds[i]
        text = test_texts[i]
        print(f'[{i}] TRUE={ID2LABEL[true]}  HAIKU={ID2LABEL[h]}  SONNET={ID2LABEL[s]}')
        print(f'     {text[:120]}')
        print()
        shown += 1
        if shown >= 4:
            break

    print('─' * 70)
    print('Sample cases where both LLMs are wrong:')
    print('─' * 70)
    shown = 0
    for i in both_wrong[:20]:
        true = test_labels[i]
        h    = haiku_preds[i]
        s    = sonnet_preds[i]
        text = test_texts[i]
        print(f'[{i}] TRUE={ID2LABEL[true]}  HAIKU={ID2LABEL[h]}  SONNET={ID2LABEL[s]}')
        print(f'     {text[:120]}')
        print()
        shown += 1
        if shown >= 4:
            break
else:
    print('Run notebook 03 first to generate LLM predictions.')